# Thailand Electricity Consumption Prediction Across Five Major Sectors
*A machine-learning project using Thailand electricity consumption data*

**Project roadmap:** 1. Business Understanding · 2. Data Understanding · 3. Data Preparation & EDA · 4. Model Development & Evaluation · 5. Model Deployment · 6. Conclusion & Limitations.

**Core machine-learning methods:** regression and regularization; validation and feature engineering; decision-tree ensembles; deployment and reproducibility. **Prediction unit:** one province × one electricity sector × one year. **Target:** annual electricity consumption (kWh).

# 1. Business Understanding — Problem Description
Electricity consumption varies across provinces and economic sectors in Thailand. Understanding these differences is important for analyzing electricity demand and supporting data-driven planning.

This project aims to develop a Machine Learning model to predict annual electricity consumption (kWh) at the provincial and sector levels in Thailand. The analysis focuses on five major electricity consumption sectors: Residential, Small Business, Medium Business, Large Business, and Specific Business.

The project uses historical electricity consumption and electricity user data from 2018 to 2023, covering 77 provinces in Thailand. The number of electricity users, province, and sector are used as input features to predict electricity consumption.

Two Machine Learning algorithms, Linear Regression and Random Forest, are applied to develop and evaluate predictive models. The project also includes data exploration, feature engineering, historical anomaly handling, and hyperparameter tuning to improve model performance.

The final model is intended to support electricity consumption analysis and scenario-based predictions across provinces and sectors. It predicts consumption based on known or estimated numbers of electricity users rather than directly forecasting future electricity demand.

# 2. Data Understanding — Data Sources & Structure
##Data Sources

This project uses historical electricity data from Thailand covering the period **2018–2023**. The datasets contain information on electricity consumers and electricity consumption across 77 provinces and 12 electricity sectors.

The data consists of two main datasets:

1. **Electricity Consumers Dataset:** Contains the number of electricity users by province, sector, and year.
2. **Electricity Consumption Dataset:** Contains the total electricity consumption in kilowatt-hours (kWh) by province, sector, and year.

An additional **Province Mapping Dataset** provides standardized province codes and English province names to support data integration and consistent analysis.

## Data Collection

The historical data was collected from annual electricity datasets for 2018–2023 and organized into separate Excel files for electricity consumers and electricity consumption.

A total of **12 annual Excel files** are used, consisting of six electricity consumer files and six electricity consumption files, together with a province mapping reference file.

The original datasets are retained before data cleaning, transformation, and sector selection.

###Dataset Structure

The original datasets contain provincial electricity information organized by year and electricity sector.

The main variables are:

| Variable | Description |
|---|---|
| `province_code` | Unique code identifying each province |
| `province_th` | Province name in Thai in the original data |
| `province_en` | Standardized English province name from the mapping dataset |
| `year` | Year of observation (2018–2023) |
| `sector` | Electricity consumption category |
| `number_of_users` | Number of electricity consumers |
| `electricity_kwh` | Total electricity consumption in kilowatt-hours |

The intended unit of analysis after data integration is **Province × Year × Sector**, where each observation represents electricity users and total electricity consumption for a specific sector within a province in a given year.

## Data Coverage

The original datasets cover:

- **Geographic coverage:** 77 provinces in Thailand
- **Time coverage:** Six years (2018–2023)
- **Electricity sectors:** 12 categories
- **Main measurements:** Number of electricity users and electricity consumption (kWh)

The full dataset is initially retained to understand the structure and coverage of electricity consumption across all sectors.

In the Data Preparation stage, five major sectors—Residential, Small Business, Medium Business, Large Business, and Specific Business—will be selected for Machine Learning.

These five sectors account for approximately **96.34% of the recorded electricity consumption** across the six-year period. The remaining sectors will be excluded from the Modeling Dataset, while the original data will be preserved for reference and comparison.

## Data Understanding Summary

The available datasets provide historical electricity consumption and consumer information across provinces, years, and electricity sectors in Thailand. The data structure supports provincial and sector-level analysis, allowing electricity consumption to be examined in relation to the number of electricity users.

The next stage, **Data Preparation**, will focus on integrating the datasets, standardizing province names, checking data quality, transforming the dataset structure, and selecting the five major electricity sectors for further analysis and Machine Learning.

# 3. Data Preparation & Exploratory Data Analysis (EDA)

### Data Preparation Overview

The Data Preparation stage focuses on transforming the original electricity datasets into a structured and reliable dataset for Machine Learning.

The original data includes electricity consumers and electricity consumption across 77 provinces, 12 electricity sectors, and six years (2018–2023).

The preparation process consists of the following steps:

1. **Data Loading:** Import the annual electricity consumer and consumption datasets from Excel files.
2. **Data Integration:** Combine annual datasets and merge electricity consumers with electricity consumption.
3. **Province Mapping:** Standardize province names in English using province codes.
4. **Data Cleaning:** Check missing values, duplicates, data types, and inconsistent records.
5. **Data Transformation:** Reshape the original datasets into a consistent Province × Year × Sector structure.
6. **Sector Selection:** Select five major electricity consumption sectors and evaluate the proportion of electricity consumption retained.
7. **Exploratory Data Analysis:** Analyze data distributions, annual trends, sector-level differences, correlations, and historical anomalies.
8. **Historical Anomaly Handling:** Use Historical Median Adjustment on flagged training values before model training.

The final prepared dataset will contain the selected five electricity sectors and serve as the basis for developing and evaluating Machine Learning models.

## 3.1 Data Loading

Load the annual Excel data for consumers and electricity sales. Keep the annual sources unchanged, preserve reporting years, and check that all required files are available before combining records.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# GitHub raw data directory
base_url = (
    "https://raw.githubusercontent.com/"
    "ChalermponBoontanapracop/"
    "Machine-Learning-Zoom-Camp/main/"
    "07-midterm-project/raw%20data/"
)

years = range(2018, 2024)

# Load all annual datasets
consumer_data = {}
usage_data = {}

for year in years:
    consumer_data[year] = pd.read_excel(
        f"{base_url}electric_consumer_{year}.xlsx"
    )

    usage_data[year] = pd.read_excel(
        f"{base_url}electric_usage_{year}.xlsx"
    )

print("Consumer datasets:", len(consumer_data))
print("Consumption datasets:", len(usage_data))

## 3.2 Data Integration

Concatenate the annual consumer and consumption tables separately. At this stage, the two source tables are not yet merged: they still represent distinct measures.

### 3.2.1 Combine Annual Consumer Datasets

In [ ]:
# Combine electricity consumer datasets (2018–2023)

consumer_frames = []

for year in years:
    df = consumer_data[year].copy()

    # Add reporting year
    df["year"] = year

    consumer_frames.append(df)

df_consumer = pd.concat(
    consumer_frames,
    ignore_index=True
)

print("Consumer Dataset Shape:", df_consumer.shape)

display(df_consumer.head())


### 3.2.2 Combine Annual Consumption Datasets

In [ ]:
# Combine electricity consumption datasets (2018–2023)

usage_frames = []

for year in years:
    df = usage_data[year].copy()

    # Add reporting year
    df["year"] = year

    usage_frames.append(df)

df_usage = pd.concat(
    usage_frames,
    ignore_index=True
)

print("Consumption Dataset Shape:", df_usage.shape)

display(df_usage.head())


### 3.2.3 Verify Data Integration

Check the number of rows contributed by each year, the reporting years, and province coverage. Unexpected differences should be investigated before reshaping the data.

In [ ]:

# Verify number of records before and after concatenation

integration_check = []

for year in years:
    integration_check.append({
        "year": year,
        "consumer_original": len(consumer_data[year]),
        "consumer_combined": (
            df_consumer["year"] == year
        ).sum(),
        "usage_original": len(usage_data[year]),
        "usage_combined": (
            df_usage["year"] == year
        ).sum()
    })

df_integration_check = pd.DataFrame(integration_check)

df_integration_check["consumer_match"] = (
    df_integration_check["consumer_original"]
    == df_integration_check["consumer_combined"]
)

df_integration_check["usage_match"] = (
    df_integration_check["usage_original"]
    == df_integration_check["usage_combined"]
)

display(df_integration_check)

assert df_integration_check[
    ["consumer_match", "usage_match"]
].all().all(), "Record count mismatch during integration"


## 3.3 Province Mapping

Standardize a stable province identifier before joining datasets. English province names improve readability in charts and in the future prediction interface, but province codes remain the reliable join key.

In [ ]:
# Load province mapping dataset

df_province_mapping = pd.read_excel(
    f"{base_url}thailand_province_mapping.xlsx"
)

print("Province Mapping Shape:",
      df_province_mapping.shape)

print("Columns:",
      df_province_mapping.columns.tolist())

display(df_province_mapping.head())


### 3.3.1 Standardize Province Code

In [ ]:
# Standardize province code column names

for df in [df_consumer, df_usage]:

    df.columns = df.columns.str.strip()

    df.rename(
        columns={
            "รหัสจังหวัด": "province_code"
        },
        inplace=True
    )

    assert "province_code" in df.columns, (
        "Province code column not found"
    )

### 3.3.2 Prepare Province Mapping

In [ ]:
# 3.3.2 Prepare Province Mapping
# Use the mapping table loaded in the previous cell
province_mapping = df_province_mapping.copy()
province_mapping.columns = province_mapping.columns.str.strip()
province_mapping = province_mapping.rename(columns={"รหัสจังหวัด": "province_code"})

# Match province codes using numeric values
province_mapping["province_code"] = pd.to_numeric(province_mapping["province_code"])
province_lookup = province_mapping.set_index("province_code")["province_en"]

# Add the English province name to both datasets
for df in [df_consumer, df_usage]:
    df["province_code"] = pd.to_numeric(df["province_code"])
    df["province_en"] = df["province_code"].map(province_lookup)

for name, df in [("Consumers", df_consumer), ("Consumption", df_usage)]:
    print(name, "provinces:", df["province_en"].nunique())
    print(name, "unmapped rows:", df["province_en"].isna().sum())


In [ ]:
# Preview datasets before Data Cleaning

print("Electricity Consumers Dataset")
print("Shape:", df_consumer.shape)
display(df_consumer.head(10))

print("\nElectricity Consumption Dataset")
print("Shape:", df_usage.shape)
display(df_usage.head(10))

## 3.4 Data Cleaning & Quality Assessment

Identify missing or non-numeric values, duplicates, negatives, zeros, and province-code inconsistencies. A zero may be valid and a missing value may reflect non-availability; neither should be automatically replaced without checking the meaning of the source data.

### 3.4.1 Column Name Standardization

In [ ]:
# 3.4.1 Column Name Standardization
import re

# Translate electricity sector names from Thai to English
sector_mapping = {
    "บ้านอยู่อาศัย": "residential",
    "กิจการขนาดเล็ก": "small_business",
    "กิจการขนาดกลาง": "medium_business",
    "กิจการขนาดใหญ่": "large_business",
    "กิจการเฉพาะอย่าง": "specific_business",
    "ไฟสำรอง": "backup_power",
    "Interruptible Rate": "interruptible_rate",
    "Interruptible Rate (IR)": "interruptible_rate_ir",
    "ไฟฟ้าสาธารณะ": "public_lighting",
    "ราชการ/รัฐวิสาหกิจ": "government_soe",
    "การสูบน้ำ": "water_pumping",
    "ไฟฟ้าชั่วคราว": "temporary_power",
    "สถานีชาร์จรถEV": "ev_charging"
}

# Rename sector columns in both datasets
for df in [df_consumer, df_usage]:
    new_names = {}

    for old_name in df.columns:
        clean_name = str(old_name).strip()
        clean_name = re.sub(r"\((ราย|kWh)\)$", "", clean_name, flags=re.IGNORECASE)
        clean_name = clean_name.strip()

        if clean_name in sector_mapping:
            new_names[old_name] = sector_mapping[clean_name]

    df.rename(columns=new_names, inplace=True)

print("Consumer columns:", df_consumer.columns.tolist())
print("Consumption columns:", df_usage.columns.tolist())

In [ ]:
# Compare sector columns between datasets

id_columns = {
    "province_code",
    "province_en",
    "province_th",
    "ชื่อจังหวัด",
    "year"
}

consumer_sectors = set(df_consumer.columns) - id_columns
usage_sectors = set(df_usage.columns) - id_columns

print("Common Sectors:")
print(sorted(consumer_sectors & usage_sectors))

print("\nConsumer-only Sectors:")
print(sorted(consumer_sectors - usage_sectors))

print("\nConsumption-only Sectors:")
print(sorted(usage_sectors - consumer_sectors))


In [ ]:
# Preview datasets after column standardization

print("Electricity Consumers Dataset")
print("Shape:", df_consumer.shape)
display(df_consumer.head(10))

print("\nElectricity Consumption Dataset")
print("Shape:", df_usage.shape)
display(df_usage.head(10))

In [ ]:
# Validate all standardized electricity sectors

id_columns = {
    "province_code",
    "province_en",
    "province_th",
    "ชื่อจังหวัด",
    "year"
}

for name, df in [
    ("Consumers", df_consumer),
    ("Consumption", df_usage)
]:
    sector_columns = [
        col for col in df.columns
        if col not in id_columns
    ]

    print(f"\n{name}")
    print("Sector Columns:", len(sector_columns))
    print("Columns:", sector_columns)

### 3.4.2 Data Types Validation

In [ ]:
# Data Types Validation

id_columns = [
    "province_code",
    "province_en",
    "year"
]

for name, df in [
    ("Consumers", df_consumer),
    ("Consumption", df_usage)
]:
    print(f"\n{name} Dataset")

    # Display column data types
    display(df.dtypes.to_frame("Data Type"))

    # Select standardized sector columns
    sector_columns = [
        col for col in sector_mapping.values()
        if col in df.columns
    ]

    # Check values that cannot be converted to numeric
    numeric_data = df[sector_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    invalid = (
        df[sector_columns].notna()
        & numeric_data.isna()
    )

    print("Non-numeric Values:", invalid.sum().sum())
    print("Missing Values:", df[sector_columns].isna().sum().sum())
    print("Province Code Type:", df["province_code"].dtype)
    print("Year Type:", df["year"].dtype)


### 3.4.3 Missing Values

In [ ]:
# Missing Values Analysis

for name, df in [
    ("Consumers", df_consumer),
    ("Consumption", df_usage)
]:
    print(f"\n{name} Dataset")

    # Calculate missing values by column
    missing_summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100
    })

    # Display missing values
    display(
        missing_summary.sort_values(
            "missing_count", ascending=False
        ).round(2)
    )

    print("Total Missing Values:", df.isna().sum().sum())
    print("Rows with Missing Values:",
          df.isna().any(axis=1).sum())

### 3.4.4 Duplicate Records

In [ ]:

# Duplicate Records Analysis

for name, df in [
    ("Consumers", df_consumer),
    ("Consumption", df_usage)
]:
    print(f"\n{name} Dataset")

    # Check completely identical rows
    print("Exact Duplicates:",
          df.duplicated().sum())

    # Check duplicate province-year combinations
    duplicate_mask = df.duplicated(
        subset=["province_code", "year"],
        keep=False
    )

    print("Duplicate Province-Year Rows:",
          duplicate_mask.sum())

    # Display duplicate records if found
    if duplicate_mask.any():
        display(
            df.loc[duplicate_mask]
            .sort_values(["province_code", "year"])
        )
    else:
        print("No duplicate Province-Year records found.")


### 3.4.5 Data Consistency & Invalid Values

In [ ]:

# Data Consistency & Invalid Values

for name, df in [
    ("Consumers", df_consumer),
    ("Consumption", df_usage)
]:
    print(f"\n{name} Dataset")

    # Check province and year consistency
    invalid_years = ~df["year"].isin(range(2018, 2024))
    missing_provinces = df["province_en"].isna()

    province_name_counts = df.groupby(
        "province_code"
    )["province_en"].nunique(dropna=False)

    inconsistent_provinces = (
        province_name_counts > 1
    ).sum()

    # Identify sector columns
    sector_columns = [
        col for col in set(sector_mapping.values())
        if col in df.columns
    ]

    # Convert values for validation only
    numeric_data = df[sector_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Check invalid numerical values
    negative_values = (numeric_data < 0).sum().sum()
    zero_values = (numeric_data == 0).sum().sum()

    non_numeric = (
        df[sector_columns].notna()
        & numeric_data.isna()
    ).sum().sum()

    # Display validation results
    summary = pd.DataFrame({
        "Check": [
            "Invalid Years",
            "Missing Province Names",
            "Inconsistent Province Codes",
            "Negative Values",
            "Zero Values",
            "Non-numeric Values"
        ],
        "Count": [
            invalid_years.sum(),
            missing_provinces.sum(),
            inconsistent_provinces,
            negative_values,
            zero_values,
            non_numeric
        ]
    })

    display(summary)


### 3.4.6 Data Quality Summary

In [ ]:
# Data Quality Summary

quality_results = []

for name, df in [
    ("Consumers", df_consumer),
    ("Consumption", df_usage)
]:
    # Identify all sector columns
    sector_columns = [
        col for col in dict.fromkeys(sector_mapping.values())
        if col in df.columns
    ]

    numeric_data = df[sector_columns].apply(
        pd.to_numeric, errors="coerce"
    )

    non_numeric = (
        df[sector_columns].notna()
        & numeric_data.isna()
    ).sum().sum()

    # Check data quality
    quality_results.append({
        "Dataset": name,
        "Records": len(df),
        "Provinces": df["province_code"].nunique(),
        "Years": df["year"].nunique(),
        "Sector Columns": len(sector_columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Keys": int(
            df.duplicated(["province_code", "year"]).sum()
        ),
        "Unmapped Provinces": int(
            df["province_en"].isna().sum()
        ),
        "Non-numeric Values": int(non_numeric),
        "Negative Values": int(
            (numeric_data < 0).sum().sum()
        ),
        "Zero Values": int(
            (numeric_data == 0).sum().sum()
        )
    })

df_quality_summary = pd.DataFrame(quality_results)

display(df_quality_summary)


## 3.5 Data Transformation — Wide to Long Format

Convert the wide sector columns into `sector` and numeric measure columns, then merge using `province_code × year × sector`. Inspect unmatched records before deciding how to handle sector-name differences (including any consumer-only IR field).

In [ ]:
# 3.5 Wide to Long Format
id_columns = ["province_code", "province_en", "year"]

# Find sector columns that exist in each dataset
consumer_sectors = []
usage_sectors = []
for sector_name in sector_mapping.values():
    if sector_name in df_consumer.columns and sector_name not in consumer_sectors:
        consumer_sectors.append(sector_name)
    if sector_name in df_usage.columns and sector_name not in usage_sectors:
        usage_sectors.append(sector_name)

# Convert sector columns into rows
# Consumer-only sectors are kept for checking; they are not automatically removed

df_consumer_long = df_consumer.melt(
    id_vars=id_columns,
    value_vars=consumer_sectors,
    var_name="sector",
    value_name="number_of_users"
)

df_usage_long = df_usage.melt(
    id_vars=id_columns,
    value_vars=usage_sectors,
    var_name="sector",
    value_name="electricity_kwh"
)

# Check that each province/year/sector appears once
merge_keys = ["province_code", "year", "sector"]
assert df_consumer_long.duplicated(merge_keys).sum() == 0
assert df_usage_long.duplicated(merge_keys).sum() == 0

# Join the two long-format datasets
usage_values = df_usage_long[merge_keys + ["electricity_kwh"]]
df_electricity = pd.merge(
    df_consumer_long,
    usage_values,
    on=merge_keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

print("Consumer long shape:", df_consumer_long.shape)
print("Consumption long shape:", df_usage_long.shape)
print("Merged shape:", df_electricity.shape)
display(df_electricity.head())

In [ ]:
# Validate merged dataset

print("Merge Status:")
display(df_electricity["_merge"].value_counts())

print("\nTotal Sectors:",
      df_electricity["sector"].nunique())

print("Unique Provinces:",
      df_electricity["province_code"].nunique())

print("Years:",
      sorted(df_electricity["year"].unique()))

print("\nMissing Values:")
display(
    df_electricity[
        ["number_of_users", "electricity_kwh"]
    ].isna().sum().to_frame("Missing Count")
)

print("\nSector Coverage:")
display(
    df_electricity.groupby(
        ["sector", "_merge"],
        observed=True
    ).size().unstack(fill_value=0)
)

## 3.6 Sector Selection

Select Residential, Small Business, Medium Business, Large Business and Specific Business. Calculate the retained share using recorded kWh across the original sectors; the resulting share is not necessarily identical in every province or year.

In [ ]:

# 3.6 Sector Selection & Coverage Analysis

selected_sectors = [
    "residential",
    "small_business",
    "medium_business",
    "large_business",
    "specific_business"
]

# Preserve all sector data
df_all_sectors = df_electricity.copy()

# Ensure electricity consumption is numeric
df_all_sectors["electricity_kwh"] = pd.to_numeric(
    df_all_sectors["electricity_kwh"],
    errors="coerce"
)

# Exclude missing values from consumption totals
df_valid_usage = df_all_sectors.dropna(
    subset=["electricity_kwh"]
)

# Calculate electricity consumption by sector
sector_coverage = (
    df_valid_usage.groupby("sector", as_index=False)
    .agg(total_kwh=("electricity_kwh", "sum"))
)

total_kwh = sector_coverage["total_kwh"].sum()

sector_coverage["share_pct"] = (
    sector_coverage["total_kwh"] / total_kwh * 100
)

sector_coverage["selection"] = np.where(
    sector_coverage["sector"].isin(selected_sectors),
    "Selected",
    "Excluded"
)

sector_coverage = sector_coverage.sort_values(
    "total_kwh", ascending=False
)

# Select five major sectors
df_selected = df_all_sectors[
    df_all_sectors["sector"].isin(selected_sectors)
].copy()

# Calculate retained and excluded electricity
selected_kwh = sector_coverage.loc[
    sector_coverage["selection"] == "Selected",
    "total_kwh"
].sum()

excluded_kwh = total_kwh - selected_kwh

coverage_summary = pd.DataFrame({
    "Category": ["Original", "Selected", "Excluded"],
    "Electricity (Billion kWh)": [
        total_kwh / 1e9,
        selected_kwh / 1e9,
        excluded_kwh / 1e9
    ],
    "Share (%)": [
        100,
        selected_kwh / total_kwh * 100,
        excluded_kwh / total_kwh * 100
    ]
})

print("Electricity Consumption by Sector")
display(sector_coverage.round(2))

print("\nCoverage Before and After Selection")
display(coverage_summary.round(2))

print("\nSelected Dataset Shape:", df_selected.shape)
print("Selected Sectors:", df_selected["sector"].nunique())


## 3.7 Exploratory Data Analysis (EDA)

How skewed are the target and user counts? How do sectors and provinces differ? Do annual trends reveal unusual observations? These are descriptive checks—not evidence of causation or proof that a model will generalize.

### 3.7.1 Descriptive Statistics

Compare count, mean, median, spread, and extreme values separately for `number_of_users` and `electricity_kwh`. Differences between the mean and median can indicate skew.

In [ ]:

# 3.7.1 Dataset Overview & Descriptive Statistics

df_eda = df_selected.copy()

# Convert sector measurements to numeric for analysis
for col in ["number_of_users", "electricity_kwh"]:
    df_eda[col] = pd.to_numeric(
        df_eda[col], errors="coerce"
    )

# Dataset overview
print("Dataset Shape:", df_eda.shape)
print("Provinces:", df_eda["province_code"].nunique())
print("Years:", sorted(df_eda["year"].unique()))
print("Sectors:", df_eda["sector"].nunique())

# Preview the dataset
display(df_eda.head(10))

# Descriptive statistics
display(
    df_eda[
        ["number_of_users", "electricity_kwh"]
    ].describe().round(2)
)

# Summary by sector
sector_stats = (
    df_eda.groupby("sector")[
        ["number_of_users", "electricity_kwh"]
    ]
    .agg(["count", "mean", "median", "min", "max"])
)

display(sector_stats.round(2))


### 3.7.2 Data Distribution

Histograms describe frequency distributions, whereas bar charts compare summary statistics by category. Log scales can reveal patterns across orders of magnitude without changing the stored training target.

In [ ]:

# 3.7.2 Data Distribution

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

variables = ["number_of_users", "electricity_kwh"]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

for i, col in enumerate(variables):

    values = df_eda[col].dropna()

    # Original distribution
    sns.histplot(
        values,
        bins=40,
        ax=axes[i, 0]
    )
    axes[i, 0].set_title(f"{col} — Original Distribution")

    # Log distribution for visualization
    log_values = np.log1p(values[values >= 0])

    sns.histplot(
        log_values,
        bins=40,
        ax=axes[i, 1]
    )
    axes[i, 1].set_title(f"{col} — Log Distribution")
    axes[i, 1].set_xlabel("log(1 + value)")

    # Boxplot
    sns.boxplot(
        x=values,
        ax=axes[i, 2]
    )
    axes[i, 2].set_title(f"{col} — Boxplot")

plt.tight_layout()
plt.show()

### 3.7.3 Distribution by Sector

Compare both typical values (median) and averages (mean). The sectors have very different sizes, so a single combined original-scale graph may obscure smaller sectors.

In [ ]:
# Electricity consumption distribution by sector

df_plot = df_eda[
    (df_eda["electricity_kwh"].notna()) &
    (df_eda["electricity_kwh"] >= 0)
].copy()

df_plot["log_electricity_kwh"] = np.log1p(
    df_plot["electricity_kwh"]
)

plt.figure(figsize=(12, 5))

sns.boxplot(
    data=df_plot,
    x="sector",
    y="log_electricity_kwh",
    order=selected_sectors
)

plt.title("Electricity Consumption Distribution by Sector")
plt.xlabel("Electricity Sector")
plt.ylabel("log(1 + Electricity Consumption in kWh)")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# Data Distribution by Sector (Bar Chart)

sector_order = [
    "residential",
    "small_business",
    "medium_business",
    "large_business",
    "specific_business"
]

sector_distribution = (
    df_eda.groupby("sector")[["number_of_users", "electricity_kwh"]]
    .agg(["mean", "median"])
    .round(2)
)

display(sector_distribution)

# Flatten column names
sector_distribution.columns = [
    f"{col[0]}_{col[1]}" for col in sector_distribution.columns
]
sector_distribution = sector_distribution.reset_index()

# Plot bar charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(
    data=sector_distribution,
    x="sector",
    y="number_of_users_mean",
    order=sector_order,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Mean Number of Users by Sector")
axes[0, 0].set_xlabel("Sector")
axes[0, 0].set_ylabel("Mean Number of Users")
axes[0, 0].tick_params(axis="x", rotation=30)

sns.barplot(
    data=sector_distribution,
    x="sector",
    y="number_of_users_median",
    order=sector_order,
    ax=axes[0, 1]
)
axes[0, 1].set_title("Median Number of Users by Sector")
axes[0, 1].set_xlabel("Sector")
axes[0, 1].set_ylabel("Median Number of Users")
axes[0, 1].tick_params(axis="x", rotation=30)

sns.barplot(
    data=sector_distribution,
    x="sector",
    y="electricity_kwh_mean",
    order=sector_order,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Mean Electricity Consumption by Sector")
axes[1, 0].set_xlabel("Sector")
axes[1, 0].set_ylabel("Mean Electricity Consumption (kWh)")
axes[1, 0].tick_params(axis="x", rotation=30)

sns.barplot(
    data=sector_distribution,
    x="sector",
    y="electricity_kwh_median",
    order=sector_order,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Median Electricity Consumption by Sector")
axes[1, 1].set_xlabel("Sector")
axes[1, 1].set_ylabel("Median Electricity Consumption (kWh)")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

### 3.7.4 Annual Electricity Consumption Trends

Aggregate the selected five sectors by year and compare changes between 2018 and 2023. These plots are descriptive; no future-year consumption is used as a predictor.

In [ ]:

# 3.7.3 Annual Electricity Consumption Trends

# Aggregate electricity consumption by year and sector
annual_sector = (
    df_eda.groupby(["year", "sector"], as_index=False)
    .agg(total_kwh=("electricity_kwh", "sum"))
)

# Convert kWh to billion kWh
annual_sector["billion_kwh"] = (
    annual_sector["total_kwh"] / 1e9
)

# Total consumption by year
annual_total = (
    annual_sector.groupby("year", as_index=False)
    .agg(billion_kwh=("billion_kwh", "sum"))
)

# Display annual summary
display(
    annual_sector.pivot(
        index="year",
        columns="sector",
        values="billion_kwh"
    ).round(2)
)

# Create charts
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Annual total consumption — Bar Chart
sns.barplot(
    data=annual_total,
    x="year",
    y="billion_kwh",
    ax=axes[0]
)

axes[0].set_title("Total Annual Electricity Consumption")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Electricity Consumption (Billion kWh)")

# Consumption trends by sector — Line Chart
sns.lineplot(
    data=annual_sector,
    x="year",
    y="billion_kwh",
    hue="sector",
    hue_order=selected_sectors,
    marker="o",
    ax=axes[1]
)

axes[1].set_title("Annual Electricity Consumption by Sector")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Electricity Consumption (Billion kWh)")
axes[1].set_xticks(sorted(df_eda["year"].unique()))
axes[1].legend(
    title="Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


### 3.7.5 Electricity Consumption by Sector

The percentage share in this section is calculated **within the five selected sectors**, unlike the original-versus-selected coverage check above.

In [ ]:

# 3.7.4 Electricity Consumption by Sector

# Aggregate total electricity consumption by sector
sector_consumption = (
    df_eda.groupby("sector", as_index=False)
    .agg(total_kwh=("electricity_kwh", "sum"))
)

# Convert to billion kWh
sector_consumption["billion_kwh"] = (
    sector_consumption["total_kwh"] / 1e9
)

# Calculate percentage share within the five selected sectors
total_consumption = sector_consumption["total_kwh"].sum()

sector_consumption["share_pct"] = (
    sector_consumption["total_kwh"]
    / total_consumption * 100
)

# Sort by consumption
sector_consumption = sector_consumption.sort_values(
    "total_kwh", ascending=False
)

display(sector_consumption.round(2))

# Visualize total consumption and percentage share
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=sector_consumption,
    x="billion_kwh",
    y="sector",
    ax=axes[0]
)
axes[0].set_title("Total Electricity Consumption by Sector (2018–2023)")
axes[0].set_xlabel("Electricity Consumption (Billion kWh)")
axes[0].set_ylabel("Sector")

sns.barplot(
    data=sector_consumption,
    x="share_pct",
    y="sector",
    ax=axes[1]
)
axes[1].set_title("Electricity Consumption Share by Sector")
axes[1].set_xlabel("Share of Selected Sectors (%)")
axes[1].set_ylabel("Sector")

plt.tight_layout()
plt.show()


### 3.7.6 Provincial Electricity Consumption

Use the same reference year (2023) for all provinces and distinguish total consumption from consumption per electricity user. High consumption is not by itself evidence of low efficiency.

In [ ]:

# 3.7.5 Provincial Electricity Consumption

# Select 2023 data
df_province_2023 = df_eda[
    df_eda["year"] == 2023
].copy()

# Aggregate electricity consumption by province
province_consumption = (
    df_province_2023.groupby(
        "province_en", as_index=False
    )
    .agg(total_kwh=("electricity_kwh", "sum"))
)

# Convert to billion kWh
province_consumption["billion_kwh"] = (
    province_consumption["total_kwh"] / 1e9
)

# Rank provinces
province_consumption = province_consumption.sort_values(
    "total_kwh", ascending=False
)

print("Top 10 Provinces")
display(province_consumption.head(10).round(3))

print("Bottom 10 Provinces")
display(province_consumption.tail(10).round(3))

# Visualize provincial electricity consumption
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    data=province_consumption.head(10),
    x="billion_kwh",
    y="province_en",
    ax=axes[0]
)
axes[0].set_title("Top 10 Provinces by Electricity Consumption (2023)")
axes[0].set_xlabel("Electricity Consumption (Billion kWh)")
axes[0].set_ylabel("Province")

sns.barplot(
    data=province_consumption.tail(10).sort_values("total_kwh"),
    x="billion_kwh",
    y="province_en",
    ax=axes[1]
)
axes[1].set_title("Bottom 10 Provinces by Electricity Consumption (2023)")
axes[1].set_xlabel("Electricity Consumption (Billion kWh)")
axes[1].set_ylabel("Province")

plt.tight_layout()
plt.show()


### 3.7.7 Users vs Electricity Consumption by Sector

Pearson measures linear association; Spearman uses ranks and assesses monotonic association. Patterns may differ when calculated within each sector versus across all sectors. Association does not establish causality.

In [ ]:
# Number of Users vs Electricity Consumption by Sector

df_scatter = df_eda[
    (df_eda["number_of_users"] > 0) &
    (df_eda["electricity_kwh"] > 0)
].copy()

g = sns.relplot(
    data=df_scatter,
    x="number_of_users",
    y="electricity_kwh",
    col="sector",
    col_wrap=3,
    col_order=selected_sectors,
    kind="scatter",
    height=3.5,
    aspect=1.2,
    alpha=0.6,
    facet_kws={
        "sharex": False,
        "sharey": False
    }
)

for ax in g.axes.flat:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(alpha=0.2)

g.set_axis_labels(
    "Number of Users (log scale)",
    "Electricity Consumption (kWh, log scale)"
)

g.set_titles("{col_name}")

g.figure.suptitle(
    "Number of Users vs Electricity Consumption, by Sector",
    y=1.03
)

plt.show()


## 3.8 Historical Anomaly Detection & Handling

### 3.8.1 Historical Anomaly Detection

Examine year-on-year changes within each province and sector. The exploratory ±50% threshold is a screening rule, not a validated definition of incorrect data. Year 2018 has no earlier year in the provided records, and a zero or missing prior-year value prevents the usual YoY calculation.

In [ ]:

# 3.7.7 Historical Anomaly Detection

# Prepare historical dataset
df_historical = df_eda.copy()

df_historical = df_historical.sort_values(
    ["province_code", "sector", "year"]
)

# Previous year's electricity consumption
df_historical["previous_kwh"] = (
    df_historical.groupby(
        ["province_code", "sector"]
    )["electricity_kwh"].shift(1)
)

# Calculate Year-over-Year percentage change
df_historical["yoy_change_pct"] = np.where(
    df_historical["previous_kwh"] > 0,
    (
        df_historical["electricity_kwh"]
        / df_historical["previous_kwh"] - 1
    ) * 100,
    np.nan
)

# Flag potential anomalies (illustrative threshold: ±50%)
threshold = 50

df_historical["potential_anomaly"] = (
    df_historical["yoy_change_pct"].abs() > threshold
)

# Summary by year
anomaly_summary = (
    df_historical.groupby("year", as_index=False)
    .agg(
        observations=("electricity_kwh", "size"),
        anomalies=("potential_anomaly", "sum")
    )
)

anomaly_summary["anomaly_pct"] = (
    anomaly_summary["anomalies"]
    / anomaly_summary["observations"] * 100
)

print("Historical Anomaly Summary")
display(anomaly_summary.round(2))

# Show potential anomalies
anomaly_records = df_historical[
    df_historical["potential_anomaly"]
].sort_values(
    "yoy_change_pct",
    key=lambda x: x.abs(),
    ascending=False
)

display(
    anomaly_records[
        [
            "province_en",
            "sector",
            "year",
            "previous_kwh",
            "electricity_kwh",
            "yoy_change_pct"
        ]
    ].head(20).round(2)
)


In [ ]:

# Historical anomalies by year and sector

anomaly_by_sector = (
    df_historical.groupby(
        ["year", "sector"], as_index=False
    )
    .agg(anomalies=("potential_anomaly", "sum"))
)

plt.figure(figsize=(12, 5))

sns.barplot(
    data=anomaly_by_sector,
    x="year",
    y="anomalies",
    hue="sector",
    hue_order=selected_sectors
)

plt.title("Potential Historical Anomalies by Year and Sector")
plt.xlabel("Year")
plt.ylabel("Number of Potential Anomalies")
plt.legend(title="Sector")

plt.tight_layout()
plt.show()


### 3.8.2 Historical Median Adjustment

Adjust flagged anomalous **training targets** where a same-province/same-sector median is available from non-flagged training observations. Preserve the original validation and test targets. A median derived from 2018–2021 should not be reused as if it were available in an earlier expanding-window training fold. No KNN adjustment is used in the main workflow.

In [ ]:
# 3.8.2 Historical Median Adjustment (training years only)

# Step 1: Split the data by year
# The 2022 and 2023 target values will not be changed
df_train = df_eda[df_eda["year"] <= 2021].copy()
df_val = df_eda[df_eda["year"] == 2022].copy()
df_test = df_eda[df_eda["year"] == 2023].copy()

# Step 2: Compare each year with the previous year
keys = ["province_code", "sector"]
df_train = df_train.sort_values(keys + ["year"])
df_train["previous_kwh"] = df_train.groupby(keys)["electricity_kwh"].shift(1)

df_train["yoy_change_pct"] = np.nan
has_previous_year = df_train["previous_kwh"] > 0

df_train.loc[has_previous_year, "yoy_change_pct"] = (
    (df_train.loc[has_previous_year, "electricity_kwh"] /
     df_train.loc[has_previous_year, "previous_kwh"] - 1) * 100
)

# Step 3: Flag large changes for review
threshold = 50
df_train["potential_anomaly"] = df_train["yoy_change_pct"].abs() > threshold

# Step 4: Find a median within each province and sector
# Do not use flagged observations in the reference median
reference = df_train[df_train["potential_anomaly"] == False].copy()
reference = reference.dropna(subset=["electricity_kwh"])
historical_median = reference.groupby(keys)["electricity_kwh"].median()

# Step 5: Keep a copy of the original training data
df_train_original = df_train.copy()
df_train_median = df_train.copy()

# Add the corresponding median to each row
median_table = historical_median.reset_index()
median_table = median_table.rename(columns={"electricity_kwh": "historical_median"})
df_train_median = df_train_median.merge(median_table, on=keys, how="left")

# Replace only flagged records when a median is available
replace_mask = df_train_median["potential_anomaly"]
replace_mask = replace_mask & df_train_median["historical_median"].notna()
df_train_median.loc[replace_mask, "electricity_kwh"] = (
    df_train_median.loc[replace_mask, "historical_median"]
)

print("Flagged rows:", df_train["potential_anomaly"].sum())
print("Adjusted rows:", replace_mask.sum())
print("Training rows:", len(df_train_median))
print("Validation rows:", len(df_val))
print("Test rows:", len(df_test))

# Show the adjusted rows and their original values
adjusted_rows = df_train_median.loc[replace_mask, keys + ["year", "electricity_kwh"]].copy()
original_values = df_train_original[keys + ["year", "electricity_kwh"]].copy()
original_values = original_values.rename(columns={"electricity_kwh": "original_kwh"})
adjusted_rows = adjusted_rows.merge(original_values, on=keys + ["year"])
display(adjusted_rows.head(20))

## 4. Model Development & Evaluation

## 4.1 Temporal Validation & Feature Preparation

Observations are split by reporting year: **2018–2021 = training**, **2022 = validation**, **2023 = test**. This is a temporal holdout, rather than the random split used in introductory exercises. The encoder is fitted to training features only. The user count used for a prediction must be known when the prediction is made; this project is therefore a conditional consumption model, not a standalone forecast of future user counts. Check whether the future `number_of_users` values are available in deployment.

In [ ]:
# 4.1 Feature Preparation
from sklearn.preprocessing import OneHotEncoder

# Use the same data split for both algorithms
train = df_train_median.copy()
val = df_val.copy()
test = df_test.copy()

features = ["province_en", "sector", "number_of_users"]
target = "electricity_kwh"
categorical_features = ["province_en", "sector"]

# Check missing values before training
for name, data in [("Train", train), ("Validation", val), ("Test", test)]:
    missing = data[features + [target]].isna().sum().sum()
    print(name, "missing values:", missing)
    assert missing == 0, f"Please review missing values in {name}"

# Separate X (features) and y (target)
X_train_raw = train[features].copy()
X_val_raw = val[features].copy()
X_test_raw = test[features].copy()
y_train = train[target].copy()
y_val = val[target].copy()
y_test = test[target].copy()

# Fit the encoder on training data only
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train_raw[categorical_features])

# We use this small function for all three datasets
# so that all feature columns have the same structure
def prepare_features(data):
    encoded_array = encoder.transform(data[categorical_features])
    column_names = encoder.get_feature_names_out(categorical_features)
    X = pd.DataFrame(encoded_array, columns=column_names, index=data.index)

    # Keep the original number of users
    X["number_of_users"] = data["number_of_users"]

    # Make one users feature for each sector
    for sector in selected_sectors:
        is_sector = (data["sector"] == sector).astype(int)
        X["users_" + sector] = data["number_of_users"] * is_sector

    return X

X_train = prepare_features(X_train_raw)
X_val = prepare_features(X_val_raw)
X_test = prepare_features(X_test_raw)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
display(X_train.head())

### 4.2 Linear Regression

This provides an interpretable linear benchmark. One-hot encoded categories and sector-specific user interactions allow a distinct fitted contribution for each sector.

#### 4.2.1 Train & Evaluate Linear Regression

In [ ]:

# 4.2.1 Linear Regression — Training & Evaluation

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Initialize and train Linear Regression
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

# Predict validation data (2022)
y_val_pred_lr = lr_model.predict(X_val)

# Calculate evaluation metrics
lr_mae = mean_absolute_error(y_val, y_val_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_lr))
lr_r2 = r2_score(y_val, y_val_pred_lr)

# Store evaluation results
lr_results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [lr_mae],
    "RMSE": [lr_rmse],
    "R2": [lr_r2]
})

display(lr_results.round(4))

# Compare actual and predicted values
lr_predictions = val[
    ["province_en", "sector", "year", "electricity_kwh"]
].copy()

lr_predictions["predicted_kwh"] = y_val_pred_lr
lr_predictions["absolute_error"] = abs(
    lr_predictions["electricity_kwh"]
    - lr_predictions["predicted_kwh"]
)

display(lr_predictions.head(10).round(2))


#### 4.2.2 Visualize Actual vs Predicted Values

In [ ]:

# Linear Regression — Actual vs Predicted

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 6))

sns.scatterplot(
    x=y_val,
    y=y_val_pred_lr,
    alpha=0.6
)

# Perfect prediction reference line
min_value = min(y_val.min(), y_val_pred_lr.min())
max_value = max(y_val.max(), y_val_pred_lr.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    color="red",
    label="Perfect Prediction"
)

plt.title("Linear Regression — Actual vs Predicted (2022)")
plt.xlabel("Actual Electricity Consumption (kWh)")
plt.ylabel("Predicted Electricity Consumption (kWh)")
plt.legend()

plt.tight_layout()
plt.show()


#### 4.2.3 Ridge Regression — Hyperparameter Tuning

Unlike `LinearRegression()`, Ridge has an L2 penalty controlled by `alpha`. Scale features inside a training-fitted pipeline so the penalty is comparable across feature magnitudes. The chosen `alpha` minimizes 2022 validation RMSE.

In [ ]:

# 4.2.2 Ridge Regression — Hyperparameter Tuning

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Alpha values to evaluate
alpha_values = [0.01, 0.1, 1, 10, 100, 1000, 10000]

ridge_results = []

for alpha in alpha_values:

    # Scale features and train Ridge Regression
    ridge_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])

    ridge_pipeline.fit(X_train, y_train)

    # Predict validation data (2022)
    y_pred = ridge_pipeline.predict(X_val)

    # Evaluation metrics
    ridge_results.append({
        "alpha": alpha,
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, y_pred)),
        "R2": r2_score(y_val, y_pred)
    })

# Compare tuning results
ridge_results = pd.DataFrame(ridge_results)

display(
    ridge_results.sort_values("RMSE").round(4)
)

# Select best alpha based on validation RMSE
best_alpha = ridge_results.loc[
    ridge_results["RMSE"].idxmin(), "alpha"
]

print("Best Alpha:", best_alpha)

# Train best Ridge Regression model
best_ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_alpha))
])

best_ridge_model.fit(X_train, y_train)

# Final validation prediction
y_val_pred_ridge = best_ridge_model.predict(X_val)

# Compare Linear Regression vs Ridge Regression
model_comparison = pd.concat([
    lr_results,
    pd.DataFrame({
        "Model": ["Ridge Regression"],
        "MAE": [mean_absolute_error(y_val, y_val_pred_ridge)],
        "RMSE": [np.sqrt(mean_squared_error(y_val, y_val_pred_ridge))],
        "R2": [r2_score(y_val, y_val_pred_ridge)]
    })
], ignore_index=True)

display(model_comparison.round(4))


In [ ]:

# Ridge Regression — Alpha vs Validation RMSE

plt.figure(figsize=(9, 5))

sns.lineplot(
    data=ridge_results,
    x="alpha",
    y="RMSE",
    marker="o"
)

plt.xscale("log")
plt.title("Ridge Regression — Hyperparameter Tuning")
plt.xlabel("Alpha (Log Scale)")
plt.ylabel("Validation RMSE (kWh)")

plt.axvline(
    best_alpha,
    color="red",
    linestyle="--",
    label=f"Best Alpha = {best_alpha:g}"
)

plt.legend()
plt.tight_layout()
plt.show()


### 4.3 Random Forest Regression

**Why Random Forest?** An ensemble of decision trees can capture nonlinear patterns and interactions without an explicitly specified linear equation, but it may struggle to extrapolate beyond consumption levels represented during training.

#### 4.3.1 Random Forest — Baseline Training & Evaluation

In [ ]:
# Random Forest Regression — Training & Evaluation

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# Initialize Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model (2018–2021)
rf_model.fit(X_train, y_train)

# Predict validation data (2022)
y_val_pred_rf = rf_model.predict(X_val)

# Evaluate model
rf_mae = mean_absolute_error(y_val, y_val_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))
rf_r2 = r2_score(y_val, y_val_pred_rf)

# Display evaluation results
rf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "MAE": [rf_mae],
    "RMSE": [rf_rmse],
    "R2": [rf_r2]
})

display(rf_results.round(4))

# Actual vs Predicted
rf_predictions = val[
    ["province_en", "sector", "year", "electricity_kwh"]
].copy()

rf_predictions["predicted_kwh"] = y_val_pred_rf

rf_predictions["absolute_error"] = abs(
    rf_predictions["electricity_kwh"]
    - rf_predictions["predicted_kwh"]
)

display(rf_predictions.head(10).round(2))

# Visualize Actual vs Predicted
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 6))

sns.scatterplot(
    x=y_val,
    y=y_val_pred_rf,
    alpha=0.6
)

min_value = min(y_val.min(), y_val_pred_rf.min())
max_value = max(y_val.max(), y_val_pred_rf.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--",
    label="Perfect Prediction"
)

plt.title("Random Forest — Actual vs Predicted (2022)")
plt.xlabel("Actual Electricity Consumption (kWh)")
plt.ylabel("Predicted Electricity Consumption (kWh)")
plt.legend()
plt.tight_layout()
plt.show()

#### 4.3.2 Random Forest — Hyperparameter Tuning

Use a reproducible random search across tree count, depth, minimum node/leaf sizes, and feature subsampling. Each candidate is fit on 2018–2021 only, then evaluated on 2022. Repeated searches on the same validation year can overfit model selection; an expanding-window cross-validation analysis would be a useful extension, with historical adjustment recomputed inside every training fold.

In [ ]:
# 4.3.2 Random Forest Hyperparameter Tuning
from sklearn.model_selection import ParameterSampler

# Try different tree settings
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.5, 0.75, 1.0, "sqrt"]
}

# Randomly select 40 combinations
param_samples = list(ParameterSampler(param_grid, n_iter=40, random_state=42))
tuning_results = []

for params in param_samples:
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    r2 = r2_score(y_val, predictions)

    result = params.copy()
    result["MAE"] = mae
    result["RMSE"] = rmse
    result["R2"] = r2
    tuning_results.append(result)

rf_tuning_results = pd.DataFrame(tuning_results)
rf_tuning_results = rf_tuning_results.sort_values("RMSE").reset_index(drop=True)
print("Top 10 settings (lowest validation RMSE first):")
display(rf_tuning_results.head(10).round(3))

In [ ]:
# Use the best parameters without converting integers to floats
# Keep the parameters from the original Python list

rmse_values = []
for result in tuning_results:
    rmse_values.append(result["RMSE"])

best_index = int(np.argmin(rmse_values))
best_params = param_samples[best_index].copy()
print("Best parameters:", best_params)

# Train the tuned model with the best parameters
best_rf_model = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
best_rf_model.fit(X_train, y_train)
y_val_pred_rf_tuned = best_rf_model.predict(X_val)

# Compare baseline and tuned Random Forest
rf_tuned_results = pd.DataFrame({
    "Model": ["Random Forest Baseline", "Random Forest Tuned"],
    "MAE": [rf_mae, mean_absolute_error(y_val, y_val_pred_rf_tuned)],
    "RMSE": [rf_rmse, np.sqrt(mean_squared_error(y_val, y_val_pred_rf_tuned))],
    "R2": [rf_r2, r2_score(y_val, y_val_pred_rf_tuned)]
})
display(rf_tuned_results.round(3))

#### 4.3.3 Random Forest — Model Evaluation

Review overall errors, sector-level errors, provincial errors, and signed residuals. Absolute kWh errors are larger for high-consumption observations by construction, so consider relative errors alongside RMSE when comparing sectors.

In [ ]:
# 4.3.3 Random Forest Evaluation (Validation 2022)

# Predict and calculate errors
y_val_pred_rf_tuned = best_rf_model.predict(X_val)
rf_mae_tuned = mean_absolute_error(y_val, y_val_pred_rf_tuned)
rf_rmse_tuned = np.sqrt(mean_squared_error(y_val, y_val_pred_rf_tuned))
rf_r2_tuned = r2_score(y_val, y_val_pred_rf_tuned)

print("MAE:", round(rf_mae_tuned, 2))
print("RMSE:", round(rf_rmse_tuned, 2))
print("R2:", round(rf_r2_tuned, 4))

# Make a table of predictions
columns = ["province_en", "sector", "year", "electricity_kwh"]
df_rf_eval = val[columns].copy()
df_rf_eval["predicted_kwh"] = y_val_pred_rf_tuned
df_rf_eval["error"] = df_rf_eval["electricity_kwh"] - df_rf_eval["predicted_kwh"]
df_rf_eval["absolute_error"] = df_rf_eval["error"].abs()

# Calculate evaluation metrics for each sector
sector_rows = []
for sector in selected_sectors:
    rows = df_rf_eval[df_rf_eval["sector"] == sector]
    if len(rows) == 0:
        continue
    actual = rows["electricity_kwh"]
    predicted = rows["predicted_kwh"]
    sector_rows.append({
        "sector": sector,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted) if len(rows) > 1 else np.nan
    })
sector_evaluation = pd.DataFrame(sector_rows)
print("Error by sector:")
display(sector_evaluation.round(2))

# Average error by province
province_evaluation = df_rf_eval.groupby("province_en")["absolute_error"].mean()
province_evaluation = province_evaluation.sort_values(ascending=False)
print("Top 10 provinces by mean absolute error:")
display(province_evaluation.head(10).to_frame("MAE").round(2))

# Three simple diagnostic charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df_rf_eval, x="electricity_kwh", y="predicted_kwh",
                hue="sector", alpha=0.6, ax=axes[0])
minimum = min(df_rf_eval["electricity_kwh"].min(), df_rf_eval["predicted_kwh"].min())
maximum = max(df_rf_eval["electricity_kwh"].max(), df_rf_eval["predicted_kwh"].max())
axes[0].plot([minimum, maximum], [minimum, maximum], "r--")
axes[0].set_title("Actual vs Predicted")
sns.barplot(data=sector_evaluation, x="RMSE", y="sector", ax=axes[1])
axes[1].set_title("RMSE by Sector")
sns.histplot(data=df_rf_eval, x="error", bins=30, ax=axes[2])
axes[2].axvline(0, color="red", linestyle="--")
axes[2].set_title("Prediction Errors")
plt.tight_layout()
plt.show()

## 4.4 Model Comparison — Baseline vs Tuned Models

Compare all four approaches on identical 2022 records; select the model with the lowest validation RMSE. Ridge is labelled separately from ordinary Linear Regression because L2 regularization changes the estimator.

In [ ]:
# Model Comparison — Four Regression Models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Define four trained models
models = {
    "Linear Regression": lr_model,
    "Linear Regression (Ridge Tuned)": best_ridge_model,
    "Random Forest": rf_model,
    "Random Forest (Tuned)": best_rf_model
}

# Evaluate all models using Validation 2022
results = []

for name, model in models.items():

    y_pred = model.predict(X_val)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, y_pred)),
        "R2": r2_score(y_val, y_pred)
    })

# Create comparison table
df_model_comparison = pd.DataFrame(results)

display(df_model_comparison.round(4))

# Visualize model performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):

    sns.barplot(
        data=df_model_comparison,
        x=metric,
        y="Model",
        ax=ax
    )

    ax.set_title(f"Model Comparison — {metric}")
    ax.set_xlabel(metric)
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

# Select best model based on Validation RMSE
best_model_name = df_model_comparison.loc[
    df_model_comparison["RMSE"].idxmin(),
    "Model"
]

final_model = models[best_model_name]

print("Selected Model:", best_model_name)

## 4.5 Final Model Testing (2023)

Evaluate the validation-selected model once on the 2023 holdout, with no further tuning based on its errors. If 2023 was examined during previous model-development experiments, disclose that the holdout is not fully untouched.

In [ ]:

# 4.5 Final Model Testing — 2023

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Selected model from Validation 2022
print("Final Selected Model:", best_model_name)

# Predict Test Dataset (2023)
y_test_pred = final_model.predict(X_test)

# Calculate final evaluation metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

# Final test evaluation
final_results = pd.DataFrame({
    "Model": [best_model_name],
    "Dataset": ["Test 2023"],
    "MAE": [test_mae],
    "RMSE": [test_rmse],
    "R2": [test_r2]
})

display(final_results.round(4))

# Compare validation and test performance
validation_results = df_model_comparison.loc[
    df_model_comparison["Model"] == best_model_name
].copy()

validation_results["Dataset"] = "Validation 2022"

performance_comparison = pd.concat([
    validation_results,
    final_results
], ignore_index=True)

display(
    performance_comparison[
        ["Model", "Dataset", "MAE", "RMSE", "R2"]
    ].round(4)
)

# Actual vs Predicted
df_test_predictions = test[
    ["province_en", "sector", "year", "electricity_kwh"]
].copy()

df_test_predictions["predicted_kwh"] = y_test_pred

df_test_predictions["absolute_error"] = abs(
    df_test_predictions["electricity_kwh"]
    - df_test_predictions["predicted_kwh"]
)

display(df_test_predictions.head(10).round(2))

# Visualize final predictions
plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=df_test_predictions,
    x="electricity_kwh",
    y="predicted_kwh",
    hue="sector",
    alpha=0.7
)

min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--",
    label="Perfect Prediction"
)

plt.title("Final Model — Actual vs Predicted (2023)")
plt.xlabel("Actual Electricity Consumption (kWh)")
plt.ylabel("Predicted Electricity Consumption (kWh)")
plt.legend()

plt.tight_layout()
plt.show()


## 4.6 Model Interpretation & Feature Importance

Permutation importance measures the change in validation RMSE when one encoded feature is shuffled. Correlated features—particularly number-of-users and sector interactions—can split importance across columns. An individual feature score is not a causal effect.

In [ ]:
# 4.6 Permutation Feature Importance (Validation 2022)
from sklearn.inspection import permutation_importance

# Measure how much the model depends on each feature
importance = permutation_importance(
    final_model,
    X_val,
    y_val,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Create a simple results table
df_importance = pd.DataFrame()
df_importance["Feature"] = X_val.columns
df_importance["Importance"] = importance.importances_mean
df_importance["Std"] = importance.importances_std

df_importance = df_importance.sort_values("Importance", ascending=False)
print("Top 15 features:")
display(df_importance.head(15).round(2))

# Higher positive importance means RMSE rises when the feature is shuffled
plt.figure(figsize=(10, 7))
sns.barplot(data=df_importance.head(15), x="Importance", y="Feature")
plt.title("Feature Importance — " + best_model_name)
plt.xlabel("Increase in RMSE (kWh)")
plt.tight_layout()
plt.show()

### 4.6.1 Interpretation notes for the report

- **Predictive association, not causation:** province and sector capture systematic differences in the observed data, not a causal effect of location or industry.
- **Feature dependence:** because `number_of_users` is also represented in sector-specific interactions, individual permutation scores can understate the importance of the underlying concept. Consider grouped permutation as a future extension.
- **Generalization:** note any difference between 2022 validation metrics and 2023 test metrics. High overall R² can hide poor performance in smaller sectors or unusual provinces.
- **Practical usage:** a prediction requires a known or projected electricity user count for the relevant province, sector, and year.

# 5. Model Deployment

Model deployment takes a trained model beyond the notebook. For this project, a Streamlit interface can collect **Province**, **Sector**, and **Number of Electricity Users**, then return predicted annual consumption in kWh. The exact same feature-encoding logic and category order used during training must be reused in inference.

**Planned workflow:** (1) freeze the selected model and feature schema; (2) save the model and encoder together with metadata; (3) provide a reproducible Python environment; (4) implement the app; (5) test with valid and invalid inputs. Streamlit is a project-specific UI choice; other deployment options include web-service APIs.

**Important:** the current selected estimator was trained on 2018–2021 for holdout evaluation. Any later refitting on 2018–2022 should be a separate, documented production-training step and must not change the reported 2023 holdout results.

In [ ]:
# 5.1 Save the selected trained model and feature configuration
# This snapshot matches the model evaluated in Section 4.
import joblib

model_bundle = {
    "model": final_model,
    "encoder": encoder,
    "features": features,
    "categorical_features": categorical_features,
    "selected_sectors": selected_sectors,
    "encoded_columns": X_train.columns.tolist(),
    "model_name": best_model_name,
    "train_years": "2018–2021",
    "validation_year": 2022,
    "test_year": 2023
}
joblib.dump(model_bundle, "electricity_model_bundle.joblib")
print("Saved electricity_model_bundle.joblib")

### 5.2 Deployment checklist

- Recreate `prepare_features()` within the inference app, using the saved encoder rather than fitting a new one; align feature columns to `encoded_columns`.
- Validate province, sector and nonnegative number of users before predicting.
- Include model limitations, units (annual kWh), and data-period information in the interface.
- Pin dependencies and document the commands required to run the app locally or deploy it.
- Do not publish raw data or private credentials in the repository.

# 6. Conclusion & Limitations

### 6.1 Key findings (complete after running the notebook)

Summarize the selected model, the chosen tuning settings, and its **2022 validation** and **2023 test** MAE, RMSE and R². Discuss error patterns by sector/province rather than treating overall R² as the only outcome.

### 6.2 Limitations

- Only six annual observations per province-sector are available (2018–2023), restricting temporal generalization.
- The five selected sectors represent most *recorded* consumption, but excluded sectors matter more in some provinces.
- Historical Median adjustment is a heuristic; flagged values may reflect real events.
- Electricity user counts must be available when the model is applied to a future period.
- The original 2023 results may have influenced earlier exploratory iterations; report this if the test period was not fully untouched.

### 6.3 Future work

Use additional years and exogenous variables where available, strengthen expanding-window validation with fold-specific cleaning, compare errors across sectors, and build a documented Streamlit prediction application.